## MASTER ACTIVATOR TEMPLATE

In [ ]:
import sys
import os
from google.colab import drive

# 1. Bind Google Drive Environment
drive.mount('/content/drive', force_remount=False)

# 2. Establish Structured Project Paths
LIB_DIR  = "/content/drive/MyDrive/radiology_ai/libs"
DATA_DIR = "/content/drive/MyDrive/radiology_ai/data"
MDL_DIR  = "/content/drive/MyDrive/radiology_ai/models"
RES_DIR  = "/content/drive/MyDrive/radiology_ai/results"
RAG_DIR  = "/content/drive/MyDrive/radiology_ai/rag_papers"
CHR_DIR  = "/content/drive/MyDrive/radiology_ai/chroma_db"

for d in [LIB_DIR, DATA_DIR, MDL_DIR, RES_DIR, RAG_DIR, CHR_DIR]:
    os.makedirs(d, exist_ok=True)

# 3. 🛡️ NON-DESTRUCTIVE ISOLATION: Hide the Drive libs from Python temporarily
sys.path = [p for p in sys.path if p != LIB_DIR]

# 4. Clear out stale cached execution imports from RAM
_stale = ['chromadb', 'gradio', 'sentence_transformers', 'pydantic',
          'huggingface_hub', 'langchain', 'transformers', 'albumentations']
for m in list(sys.modules.keys()):
    if any(s in m for s in _stale):
        del sys.modules[m]

# 5. Import base system dependencies safely from Colab's native system
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import transformers
import huggingface_hub
print(f"📦 System Core: transformers v{transformers.__version__} initialized natively from {transformers.__file__} ✅")

# 6. Safely restore Drive libraries for downstream LangChain usage
if LIB_DIR not in sys.path:
    sys.path.append(LIB_DIR)

# 7. Hardware Acceleration Verification
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Environment Configured | Target Compute Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"🚀 Active Accelerator: {torch.cuda.get_device_name(0)}")
    print(f"💾 Dedicated VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU")

##  Download 20 Real Europe PMC PDFs

In [ ]:
import urllib.request
import time
import os
import xml.etree.ElementTree as ET

PAPERS = [
    ("PMC4345928", "Pneumonia"), ("PMC7187846", "COVID-19"),
    ("PMC6110037", "Nodule"), ("PMC4522053", "Tuberculosis"),
    ("PMC4700002", "Atelectasis"), ("PMC3339339", "Effusion"),
    ("PMC5374950", "Cardiomegaly"), ("PMC6393282", "Emphysema"),
    ("PMC4159553", "Fibrosis"), ("PMC7418140", "Pneumothorax"),
    ("PMC5504479", "Mass"), ("PMC4519692", "Infiltration"),
    ("PMC4418061", "Edema"), ("PMC3908954", "Consolidation"),
    ("PMC5826833", "Deep Learning"), ("PMC6937416", "Deep Learning"),
    ("PMC7294175", "Grad-CAM XAI"), ("PMC7751753", "COVID-19"),
    ("PMC6117195", "Nodule"), ("PMC5526827", "Pleural_Thickening"),
]

def fetch_pmc_text(pmcid):
    url = f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML"
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0', 'Accept': 'application/xml'})
        with urllib.request.urlopen(req, timeout=30) as r:
            xml_bytes = r.read()
        root = ET.fromstring(xml_bytes)
        for ref_sec in root.findall('.//{*}ref-list'):
            ref_sec.clear()
        texts = [elem.text.strip() for elem in root.iter() if elem.text and elem.text.strip()]
        texts += [elem.tail.strip() for elem in root.iter() if elem.tail and elem.tail.strip()]
        full_text = ' '.join(texts)
        return full_text if len(full_text) > 300 else None
    except Exception:
        return None

def fetch_pmc_abstract(pmcid):
    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pmc&id={pmcid}&rettype=abstract&retmode=text"
    try:
        req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.read().decode('utf-8', errors='ignore')
    except:
        return None

print("⬇️ Fetching clinical literature...\n")
failed = []
for pmcid, disease in PAPERS:
    cache_path = f"{RAG_DIR}/{pmcid}.txt"
    if os.path.exists(cache_path):
        print(f"  ✅ {pmcid} ({disease}) — Loaded from cache")
    else:
        text = fetch_pmc_text(pmcid)
        if text:
            print(f"  ✅ {pmcid} ({disease}) — Full text downloaded")
        else:
            text = fetch_pmc_abstract(pmcid)
            if text and len(text) > 100:
                print(f"  ⚠️ {pmcid} ({disease}) — Abstract only")
            else:
                print(f"  ❌ {pmcid} ({disease}) — Failed")
                failed.append(pmcid)
                continue
        with open(cache_path, 'w', encoding='utf-8') as f:
            f.write(text)
    time.sleep(0.3)

print(f"\n✅ Data Acquisition Complete. Failed: {len(failed)}")

## Advanced NLP & Contextual Chunking  

In [ ]:
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("⚙️ Processing raw clinical texts...", flush=True)
raw_documents = []
for pmcid, disease in PAPERS:
    cache_path = f"{RAG_DIR}/{pmcid}.txt"
    if os.path.exists(cache_path):
        with open(cache_path, 'r', encoding='utf-8') as f:
            raw_documents.append(Document(
                page_content=f.read().strip()[:15000],
                metadata={"pmcid": pmcid, "disease": disease}
            ))

# Token-optimized splitting to prevent LLM truncation
splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=50)
chunks = splitter.split_documents(raw_documents)

# 🧠 ADVANCED RAG: Contextual Metadata Injection
for chunk in chunks:
    disease_context = chunk.metadata.get('disease', 'Clinical Study')
    chunk.page_content = f"Clinical Topic: {disease_context}. Findings: {chunk.page_content}"

print(f"✂️ Successfully partitioned and context-enriched {len(chunks)} semantic chunks.")

##  Custom Embedding Bridge & ChromaDB Storage

In [ ]:
import torch
from langchain_community.vectorstores import Chroma
from sentence_transformers import SentenceTransformer

# Custom PyTorch Embedding Bridge (Bypasses Pydantic & LangChain bugs)
print("🚀 Initializing Custom Sentence-Transformer Bridge...", flush=True)
class DirectHuggingFaceEmbeddings:
    def __init__(self, model_name: str, device: str):
        self.model = SentenceTransformer(model_name, device=device)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.encode(texts, convert_to_numpy=True).tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.model.encode(text, convert_to_numpy=True).tolist()

    def __call__(self, text: str) -> list[float]:
        return self.embed_query(text)

embeddings = DirectHuggingFaceEmbeddings("sentence-transformers/all-MiniLM-L6-v2", DEVICE)

print("📂 Constructing ChromaDB Vector Space inside Google Drive...", flush=True)
# Build and store directly inside your permanent Drive folder
vectordb = Chroma.from_documents(chunks, embeddings, persist_directory=CHR_DIR)

# Force the physical write configuration to save on disk
vectordb.persist()
print(f"✅ ChromaDB built and secured with {vectordb._collection.count()} vectors inside your Drive!")

##  Vector Store Retrieval Evaluation  

In [ ]:
import json
import os

print("📊 Advanced Retrieval Evaluation (Top 5):")
queries = [
    ("pneumonia bacterial consolidation", "Pneumonia"),
    ("COVID-19 viral ground glass", "COVID-19"),
    ("pulmonary nodule Fleischner management", "Nodule"),
    ("pleural effusion costophrenic angle", "Effusion"),
    ("deep learning neural network x-ray", "Deep Learning"),
]

hits = 0
for q, expected in queries:
    res = vectordb.similarity_search(q, k=5)  # Points to your fresh Chroma database instance
    got = list(set([r.metadata.get('disease') for r in res]))
    hit = expected in got
    if hit: hits += 1
    print(f"  {'✅' if hit else '❌'} Expected={expected.ljust(15)} | Found: {got}")

precision = (hits / len(queries)) * 100
print(f"\n📈 Final Retrieval Precision: {precision:.1f}%")

# Save Metrics
eval_data = {'retrieval_precision': precision, 'chunks': len(chunks)}
with open(f"{RES_DIR}/rag_eval_final.json", 'w') as f:
    json.dump(eval_data, f)
print(f"💾 Metrics saved to: {RES_DIR}")

## Core LLM Initialization & Custom Bridge  

In [ ]:
import sys
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from typing import Optional, List, Any

# Ensure priority pathing is set for core classes
LIB_DIR = "/content/drive/MyDrive/radiology_ai/libs"
if LIB_DIR not in sys.path:
    sys.path.append(LIB_DIR)

# Safe fallback for the foundational LLM class
try:
    from langchain_core.language_models.llms import LLM
except ImportError:
    !pip install langchain-core --quiet
    from langchain_core.language_models.llms import LLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Core Model Weights
print("⬇️ Loading flan-t5-large execution weights...", flush=True)
tok = AutoTokenizer.from_pretrained("google/flan-t5-large")
lm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to(DEVICE)

# 2. Custom PyTorch Bridge (Bypasses huggingface pipeline bugs)
print("🚀 Initializing Custom PyTorch-to-LangChain Bridge...", flush=True)
class DirectFlanT5(LLM):
    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs: Any) -> str:
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        outputs = lm.generate(**inputs, max_new_tokens=256)
        return tok.decode(outputs[0], skip_special_tokens=True)

    @property
    def _llm_type(self) -> str:
        return "direct_flan_t5"

llm = DirectFlanT5()
print("✅ Custom LLM Bridge Ready!")

## Modern LCEL Chain & Clinical Inference  

In [ ]:
import sys
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Define Clinical AI Instructions
PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an expert radiologist AI assistant.
Use the published medical evidence below to answer the question.
Cover: imaging findings, clinical significance, and management.

Evidence:
{context}

Question: {question}

Answer:"""
)

print("⚙️ Constructing modern LCEL RAG architecture...", flush=True)

# Helper function to format retrieved documents cleanly into a single context string
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

# Create an optimized MMR retriever from our Cell 2.3 vector database
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 3})

# Modern LCEL Chain construction (Bypasses the broken 'langchain.chains' module entirely)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
)
print("✅ High-Precision Custom LCEL RAG Chain Online!")

# 3. Run Clinical Evaluation
print("\n🔍 Evaluating retrieval synthesis request...", flush=True)
query = "What does pneumonia look like on chest X-ray?"

# Fetch source docs separately for verification display
source_documents = retriever.invoke(query)
result = rag_chain.invoke(query)

print("="*60)
print(f"🤖 AI Radiologist Diagnosis:\n{result}")
print("-" * 60)
print(f"📚 Verification Sources: {list(set([d.metadata.get('disease', 'Unlabeled') for d in source_documents]))}")
print("="*60)